In [1]:
# === SESSION BOOTSTRAP ===  (SWITCH RUNTIME TO GPU before running this notebook)
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT="/content/drive/MyDrive/UAV_TRUST_Research"; REPO=f"{PARENT}/uav-trust-research"
for fn in (".gitconfig",".git-credentials"):
    p=os.path.join(PARENT,fn)
    if os.path.exists(p): subprocess.run(f'cp "{p}" /root/{fn}',shell=True)
subprocess.run("git config --global credential.helper store",shell=True)
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0,REPO) if REPO not in sys.path else None; print("cwd:",os.getcwd())
else: print("run 00_setup first")

Mounted at /content/drive
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install pandas numpy pyarrow scikit-learn matplotlib --quiet
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

torch 2.11.0+cu128 | CUDA available: True


In [3]:
# Representation-invariance test: does the conformal coverage collapse survive a DIFFERENT representation
# of the SAME flows? Train a 1D-CNN on raw packet sequences (UAV-CAS time-series), same families held out,
# run ONLY the conformal coverage panel. Claim = coverage finding only (not mechanism/TSFS).
CAS_TS_PARQUET="/content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research/data/uav_cas/uav_cas_ts.parquet"
CAS_STAT_CMP="reports/14_uavcas_panel_raw.csv"   # tabular UAV-CAS coverage, for side-by-side comparison
NORMAL_KEY="benign"
L=1105                       # pad/truncate length (95th pct from notebook 15)
SUBSAMPLE_BENIGN=20000       # cap benign; keep all attack flows (rare families intact)
CFG={"seeds":[0,1,2],"alpha":0.10,"epochs":12,"batch":256,"lr":1e-3,"weight_decay":1e-5,
     "normal_fracs":{"train":0.60,"cal":0.20,"test_seen":0.10,"test_shift":0.10},
     "family_fracs":{"train":0.60,"cal":0.20,"test_seen":0.20},
     "fig_dir":"figures","report_dir":"reports"}
import os
for d in [CFG["fig_dir"],CFG["report_dir"]]: os.makedirs(d,exist_ok=True)
print("configured; L =",L)

configured; L = 1105


In [4]:
import numpy as np, pandas as pd, gc, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from src.trust import conformal_qhat, top_label_ece
dev="cuda" if torch.cuda.is_available() else "cpu"; print("device:",dev)
if dev=="cpu": print("WARNING: no GPU detected. This will be slow. Switch runtime to GPU.")

ts=pd.read_parquet(CAS_TS_PARQUET); print("loaded ts flows:",len(ts))
lab=ts["Label"].astype(str); nv=[v for v in ts["Label"].unique() if str(v).lower()==NORMAL_KEY][0]
# stratified subsample: keep all attacks, cap benign
rng=np.random.default_rng(42)
idx_norm=np.where(lab.values==nv)[0]
if len(idx_norm)>SUBSAMPLE_BENIGN: idx_norm=rng.choice(idx_norm,SUBSAMPLE_BENIGN,replace=False)
idx_atk=np.where(lab.values!=nv)[0]
keep=np.sort(np.concatenate([idx_norm,idx_atk])); ts=ts.iloc[keep].reset_index(drop=True)
fams=[v for v in ts["Label"].unique() if v!=nv]
print("after subsample:",len(ts),"| normal:",repr(nv),"| families:",fams)
print(ts["Label"].value_counts().to_string())

device: cuda
loaded ts flows: 92131
after subsample: 53542 | normal: 'Benign' | families: ['DoS', 'DDoS', 'Blackhole', 'Wormhole', 'Replay']
Label
Benign       20000
Blackhole    12116
Wormhole     10862
DDoS          7316
Replay        2025
DoS           1223


In [5]:
# Build padded 3-channel tensor (packet size, inter-arrival time, direction) once, standardized on valid packets.
def to_channels(pt,ps,pdir,L):
    pt=np.asarray(pt,dtype=np.float64); ps=np.asarray(ps,dtype=np.float64); pdir=np.asarray(pdir,dtype=np.float64)
    n=min(len(pt),L); arr=np.zeros((L,3),dtype=np.float32); m=np.zeros(L,dtype=np.float32)
    if n>0:
        iat=np.zeros(n)
        if n>1: iat[1:]=np.clip(np.diff(pt[:n]),0,None)
        arr[:n,0]=np.log1p(ps[:n]); arr[:n,1]=np.log1p(iat); arr[:n,2]=pdir[:n]; m[:n]=1.0
    return arr,m

N=len(ts); X=np.zeros((N,L,3),dtype=np.float32); M=np.zeros((N,L),dtype=np.float32)
pt_c,ps_c,pd_c=ts["packet_time"].values,ts["packet_size"].values,ts["packet_dir"].values
for i in range(N):
    X[i],M[i]=to_channels(pt_c[i],ps_c[i],pd_c[i],L)
    if (i+1)%20000==0: print("  built",i+1)
y=(ts["Label"].values!=nv).astype(np.int64); fam=ts["Label"].values
# standardize each channel over valid (non-pad) positions, globally
vb=M.astype(bool)
for ch in range(3):
    vals=X[:,:,ch][vb]; mu,sd=float(vals.mean()),float(vals.std())+1e-6
    X[:,:,ch]=(X[:,:,ch]-mu)/sd; X[:,:,ch][~vb]=0.0
X=np.transpose(X,(0,2,1)).copy()  # (N,3,L) for Conv1d
print("tensor:",X.shape, "mem %.0f MB"%(X.nbytes/1e6)); del pt_c,ps_c,pd_c; gc.collect()

  built 20000
  built 40000
tensor: (53542, 3, 1105) mem 710 MB


0

In [6]:
# Compact 1D-CNN with strided pooling (handles long sequences efficiently)
class PacketCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv1d(3,32,7,stride=2,padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32,64,5,stride=2,padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64,64,5,stride=2,padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64,64,3,stride=2,padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.head=nn.Linear(64,1)
    def forward(self,x): return self.head(self.net(x).squeeze(-1)).squeeze(-1)

def split_idx(fam,held_out,seed):
    seen=[f for f in fams if f!=held_out]
    def sp(ix,fr,sd):
        ix=np.array(ix); r=np.random.default_rng(sd); r.shuffle(ix); out={}; s=0; ks=list(fr)
        for i,k in enumerate(ks):
            e=len(ix) if i==len(ks)-1 else s+int(round(fr[k]*len(ix))); out[k]=ix[s:e]; s=e
        return out
    tr=[];ca=[];se=[];sh=[]
    ns=sp(np.where(fam==nv)[0],CFG["normal_fracs"],seed); tr+=list(ns["train"]);ca+=list(ns["cal"]);se+=list(ns["test_seen"]);sh+=list(ns["test_shift"])
    for j,f in enumerate(seen):
        fs=sp(np.where(fam==f)[0],CFG["family_fracs"],seed+j+1); tr+=list(fs["train"]);ca+=list(fs["cal"]);se+=list(fs["test_seen"])
    sh+=list(np.where(fam==held_out)[0])
    return (np.array(sorted(a)) for a in (tr,ca,se,sh))

def train_probs(tr,ca,se,sh,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Xt=torch.tensor(X[tr]); yt=torch.tensor(y[tr],dtype=torch.float32)
    pos=float(yt.mean()); w=torch.tensor([(1-pos)/max(pos,1e-3)]).to(dev)
    dl=DataLoader(TensorDataset(Xt,yt),batch_size=CFG["batch"],shuffle=True)
    net=PacketCNN().to(dev); opt=torch.optim.Adam(net.parameters(),lr=CFG["lr"],weight_decay=CFG["weight_decay"])
    lossf=nn.BCEWithLogitsLoss(pos_weight=w)
    net.train()
    for ep in range(CFG["epochs"]):
        for xb,yb in dl:
            xb,yb=xb.to(dev),yb.to(dev); opt.zero_grad(); loss=lossf(net(xb),yb); loss.backward(); opt.step()
    net.eval()
    def prob(ix):
        out=[]
        with torch.no_grad():
            for s in range(0,len(ix),512):
                xb=torch.tensor(X[ix[s:s+512]]).to(dev); out.append(torch.sigmoid(net(xb)).cpu().numpy())
        return np.concatenate(out) if len(ix) else np.array([])
    p=(prob(ca),prob(se),prob(sh)); del net,Xt,yt; gc.collect(); torch.cuda.empty_cache() if dev=="cuda" else None
    return p

In [ ]:
# Held-out protocol on the SEQUENCE representation -> conformal coverage panel only
def cov_detail(p,yv,qh):
    p=np.asarray(p);yv=np.asarray(yv);ia=(1-p)<=qh;ino=p<=qh; inset=np.where(yv==1,ia,ino)
    return float(inset.mean()), float(np.mean([inset[yv==k].mean() for k in np.unique(yv)])), float((ia.astype(int)+ino.astype(int)).mean())
rows=[]
for F in fams:
    for seed in CFG["seeds"]:
        tr,ca,se,sh=split_idx(fam,F,seed)
        p_ca,p_se,p_sh=train_probs(tr,ca,se,sh,seed)
        qh=conformal_qhat(p_ca,y[ca],alpha=CFG["alpha"])
        m_sh,b_sh,sz_sh=cov_detail(p_sh,y[sh],qh); m_se,_,_=cov_detail(p_se,y[se],qh)
        rows.append({"held_out":str(F),"seed":seed,
            "shift_balacc":balanced_accuracy_score(y[sh],(p_sh>=.5).astype(int)),
            "shift_cov_marg":m_sh,"shift_cov_bal":b_sh,"shift_setsize":sz_sh,"seen_cov_marg":m_se})
        print(f"  seq {F} seed{seed}: balacc={rows[-1]['shift_balacc']:.3f} cov_marg={m_sh:.3f} cov_bal={b_sh:.3f}")
    print(" SEQUENCE",F,"done")
Q=pd.DataFrame(rows); Q.to_csv(os.path.join(CFG["report_dir"],"16_uavcas_ts_coverage_raw.csv"),index=False)
print("in-distribution (seen) marginal coverage, sequence model: %.3f +/- %.3f"%(Q["seen_cov_marg"].mean(),Q["seen_cov_marg"].std()))

  seq DoS seed0: balacc=0.998 cov_marg=1.000 cov_bal=1.000
  seq DoS seed1: balacc=0.994 cov_marg=0.999 cov_bal=0.999
  seq DoS seed2: balacc=0.998 cov_marg=0.998 cov_bal=0.999
 SEQUENCE DoS done
  seq DDoS seed0: balacc=0.863 cov_marg=1.000 cov_bal=1.000
  seq DDoS seed1: balacc=0.883 cov_marg=1.000 cov_bal=0.999
  seq DDoS seed2: balacc=0.820 cov_marg=0.989 cov_bal=0.973
 SEQUENCE DDoS done
  seq Blackhole seed0: balacc=0.500 cov_marg=0.713 cov_bal=0.833
  seq Blackhole seed1: balacc=0.499 cov_marg=0.712 cov_bal=0.832
  seq Blackhole seed2: balacc=0.499 cov_marg=0.705 cov_bal=0.827
 SEQUENCE Blackhole done
  seq Wormhole seed0: balacc=0.499 cov_marg=0.731 cov_bal=0.840
  seq Wormhole seed1: balacc=0.500 cov_marg=0.737 cov_bal=0.844
  seq Wormhole seed2: balacc=0.497 cov_marg=0.731 cov_bal=0.839
 SEQUENCE Wormhole done
  seq Replay seed0: balacc=0.501 cov_marg=0.876 cov_bal=0.877


In [ ]:
# Side-by-side: does coverage collapse on the SAME families under the sequence representation as under tabular?
seq=Q.groupby("held_out").agg(seq_balacc=("shift_balacc","mean"),seq_cov_marg=("shift_cov_marg","mean"),seq_cov_bal=("shift_cov_bal","mean")).round(3)
try:
    tab=pd.read_csv(CAS_STAT_CMP).groupby("held_out").agg(tab_balacc=("shift_balacc","mean"),tab_cov_marg=("shift_cov_marg","mean")).round(3)
    comp=tab.join(seq)
except Exception as e:
    print("tabular comparison csv not found; showing sequence only", e); comp=seq
print("=== Coverage under two representations of the SAME flows (target 0.90) ===")
print(comp.to_string())
comp.to_csv(os.path.join(CFG["report_dir"],"16_uavcas_representation_compare.csv"))
# does the ranking of which families collapse agree across representations?
if "tab_cov_marg" in comp and comp["tab_cov_marg"].notna().all():
    print("\nSpearman(tabular coverage, sequence coverage) across families = %.3f"%spearmanr(comp["tab_cov_marg"],comp["seq_cov_marg"]).correlation)
    print("families below 0.80 -- tabular: %d/5, sequence: %d/5"%((comp["tab_cov_marg"]<0.8).sum(),(comp["seq_cov_marg"]<0.8).sum()))

In [ ]:
# FIGURE: tabular vs sequence marginal coverage per family
import matplotlib.pyplot as plt
o=comp.reset_index()
x=np.arange(len(o)); w=0.38
fig,ax=plt.subplots(figsize=(9,4.2))
if "tab_cov_marg" in o: ax.bar(x-w/2,o["tab_cov_marg"],w,label="tabular (47 features, XGBoost)",color="#264653")
ax.bar(x+w/2,o["seq_cov_marg"],w,label="sequence (raw packets, 1D-CNN)",color="#e76f51")
ax.axhline(0.90,ls="--",color="gray"); ax.set_xticks(x); ax.set_xticklabels(o["held_out"],rotation=20,ha="right")
ax.set_ylabel("shift marginal coverage"); ax.set_ylim(0,1.05)
ax.set_title("Coverage collapse is representation-invariant: same flows, two representations"); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(CFG["fig_dir"],"16_representation_invariance.png"),dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
!git add reports/ figures/ notebooks/
!git commit -m "16 representation-invariance: 1D-CNN on raw UAV-CAS packet sequences, same families held out; conformal coverage panel vs tabular representation"
!git push origin main